In [1]:
import os
print(os.listdir("data"))

['cleaned_crime_long.csv', 'borough_descriptive_stats.csv', 'category_descriptive_stats.csv', 'arima_comparison.csv', 'timeseries_comparison.csv', 'borough_clusters_final.csv', 'future_risk_predictions.csv', 'borough_forecast_validation_all32.csv', 'borough_future_forecasts_Jul-Dec_2026.csv', 'borough_future_forecasts_pivot.csv', '.ipynb_checkpoints', 'category_forecast_validation.csv', 'borough_category_forecast_validation.csv', 'ensemble_results.csv', 'category_trend_aug_dec_2026.csv', 'category_full_forecast_aug_dec_2026.csv']


In [2]:
import pandas as pd
from statsmodels.tsa.statespace.sarimax import SARIMAX

df = pd.read_csv("data/cleaned_crime_long.csv")
df["Date"] = pd.to_datetime(df["Date"])

london_monthly = df.groupby("Date")["CrimeCount"].sum().reset_index()
print(london_monthly.tail())

          Date  CrimeCount
186 2025-10-01       76963
187 2025-11-01       76013
188 2025-12-01       73393
189 2026-01-01       70305
190 2026-02-01       67469


In [3]:
print(df["Date"].min(), "to", df["Date"].max())
print(df.shape)

2010-04-01 00:00:00 to 2026-02-01 00:00:00
(195175, 7)


In [5]:
newer = pd.read_csv("MPS_Borough_Level_Crime _Most_Recent_24_months_new.csv")
print(newer.columns.tolist())
newer.head()

['Group', 'SubGroup', 'BOCU', '202407', '202408', '202409', '202410', '202411', '202412', '202501', '202502', '202503', '202504', '202505', '202506', '202507', '202508', '202509', '202510', '202511', '202512', '202601', '202602', '202603', '202604', '202605', '202606']


,Group,SubGroup,BOCU,202407,202408,202409,202410,202411,202412,202501,...,202509,202510,202511,202512,202601,202602,202603,202604,202605,202606
0,ARSON AND CRIMINAL DAMAGE,ARSON,Barking and Dagenham,3,10,9,5,7,9,7,...,5,6,8,7,0,3,3,5,4,7
1,ARSON AND CRIMINAL DAMAGE,CRIMINAL DAMAGE,Barking and Dagenham,131,114,80,103,96,89,106,...,99,103,106,125,104,97,116,109,131,112
2,BURGLARY,BURGLARY BUSINESS AND COMMUNITY,Barking and Dagenham,21,26,32,29,25,22,17,...,23,16,23,23,27,26,22,22,24,19
3,BURGLARY,RES BURGLARY OF A HOME,Barking and Dagenham,28,35,31,48,40,60,56,...,31,33,64,52,47,40,44,27,48,35
4,BURGLARY,RES BURGLARY OF UNCONNECTED BUILDING,Barking and Dagenham,8,12,18,14,11,23,15,...,12,22,15,23,7,17,16,13,10,15


In [6]:
# Sum across all boroughs and categories, for Mar-Jun 2026
mar_jun_cols = ["202603", "202604", "202605", "202606"]

london_actual_mar_jun = newer[mar_jun_cols].sum()
print(london_actual_mar_jun)

202603    75137
202604    73990
202605    78652
202606    79894
dtype: int64


In [7]:
comparison = pd.DataFrame({
    "Month": ["2026-03", "2026-04", "2026-05", "2026-06"],
    "Actual": [75137, 73990, 78652, 79894],
    "Forecast": [73103, 71817, 76096, 75829]
})
comparison["Pct_Error"] = (abs(comparison["Actual"] - comparison["Forecast"]) / comparison["Actual"] * 100).round(1)

print(comparison)
comparison.to_csv("total_actual_vs_forecast.csv", index=False)
print("Saved total_actual_vs_forecast.csv")

     Month  Actual  Forecast  Pct_Error
0  2026-03   75137     73103        2.7
1  2026-04   73990     71817        2.9
2  2026-05   78652     76096        3.2
3  2026-06   79894     75829        5.1
Saved total_actual_vs_forecast.csv


In [8]:
import os
print(os.listdir("data"))

['cleaned_crime_long.csv', 'borough_descriptive_stats.csv', 'category_descriptive_stats.csv', 'arima_comparison.csv', 'timeseries_comparison.csv', 'borough_clusters_final.csv', 'future_risk_predictions.csv', 'borough_forecast_validation_all32.csv', 'borough_future_forecasts_Jul-Dec_2026.csv', 'borough_future_forecasts_pivot.csv', '.ipynb_checkpoints', 'category_forecast_validation.csv', 'borough_category_forecast_validation.csv', 'ensemble_results.csv', 'category_trend_aug_dec_2026.csv', 'category_full_forecast_aug_dec_2026.csv']


In [9]:
mar_jun_cols = ["202603", "202604", "202605", "202606"]

borough_actuals = newer.groupby("BOCU")[mar_jun_cols].sum().reset_index()
borough_actuals.columns = ["Borough", "Actual_2026-03", "Actual_2026-04", "Actual_2026-05", "Actual_2026-06"]

print(borough_actuals.shape)
borough_actuals.head()

(33, 5)


,Borough,Actual_2026-03,Actual_2026-04,Actual_2026-05,Actual_2026-06
0,Barking and Dagenham,1761,1650,1830,1870
1,Barnet,2651,2352,2519,2590
2,Bexley,1405,1237,1421,1489
3,Brent,2805,2870,3022,2835
4,Bromley,2056,2029,2123,2103


In [10]:
print(borough_actuals["Borough"].tolist())

['Barking and Dagenham', 'Barnet', 'Bexley', 'Brent', 'Bromley', 'Camden', 'Croydon', 'Ealing', 'Enfield', 'Greenwich', 'Hackney', 'Hammersmith and Fulham', 'Haringey', 'Harrow', 'Havering', 'Hillingdon', 'Hounslow', 'Islington', 'Kensington and Chelsea', 'Kingston upon Thames', 'Lambeth', 'Lewisham', 'Merton', 'Newham', 'Redbridge', 'Richmond upon Thames', 'Southwark', 'Sutton', 'Tower Hamlets', 'Unknown', 'Waltham Forest', 'Wandsworth', 'Westminster']


In [11]:
borough_actuals = borough_actuals[borough_actuals["Borough"] != "Unknown"]
print(borough_actuals.shape)  # should now be (32, 5)
borough_actuals.head()

(32, 5)


,Borough,Actual_2026-03,Actual_2026-04,Actual_2026-05,Actual_2026-06
0,Barking and Dagenham,1761,1650,1830,1870
1,Barnet,2651,2352,2519,2590
2,Bexley,1405,1237,1421,1489
3,Brent,2805,2870,3022,2835
4,Bromley,2056,2029,2123,2103


In [13]:
from statsmodels.tsa.statespace.sarimax import SARIMAX

boroughs = sorted(borough_actuals["Borough"].unique())
forecast_results = []

for borough in boroughs:
    borough_hist = df[df["Borough"] == borough].groupby("Date")["CrimeCount"].sum().reset_index()
    borough_hist = borough_hist.sort_values("Date")
    ts = borough_hist.set_index("Date")["CrimeCount"]
    
    try:
        model = SARIMAX(ts, order=(1,1,0), seasonal_order=(1,1,0,12),
                         enforce_stationarity=False, enforce_invertibility=False)
        fit = model.fit(disp=False)
        forecast = fit.forecast(steps=4)
        forecast_results.append({
            "Borough": borough,
            "Forecast_2026-03": round(forecast.iloc[0]),
            "Forecast_2026-04": round(forecast.iloc[1]),
            "Forecast_2026-05": round(forecast.iloc[2]),
            "Forecast_2026-06": round(forecast.iloc[3]),
        })
        print(f"Done: {borough}")
    except Exception as e:
        print(f"FAILED: {borough} - {e}")

forecast_df = pd.DataFrame(forecast_results)
print(forecast_df.shape)
forecast_df.head()

/opt/conda/lib/python3.12/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency MS will be used.
  self._init_dates(dates, freq)
/opt/conda/lib/python3.12/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency MS will be used.
  self._init_dates(dates, freq)
/opt/conda/lib/python3.12/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency MS will be used.
  self._init_dates(dates, freq)
/opt/conda/lib/python3.12/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency MS will be used.
  self._init_dates(dates, freq)


Done: Barking and Dagenham
Done: Barnet


/opt/conda/lib/python3.12/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency MS will be used.
  self._init_dates(dates, freq)
/opt/conda/lib/python3.12/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency MS will be used.
  self._init_dates(dates, freq)
/opt/conda/lib/python3.12/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency MS will be used.
  self._init_dates(dates, freq)
/opt/conda/lib/python3.12/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency MS will be used.
  self._init_dates(dates, freq)


Done: Bexley
Done: Brent


/opt/conda/lib/python3.12/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency MS will be used.
  self._init_dates(dates, freq)
/opt/conda/lib/python3.12/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency MS will be used.
  self._init_dates(dates, freq)
/opt/conda/lib/python3.12/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency MS will be used.
  self._init_dates(dates, freq)
/opt/conda/lib/python3.12/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency MS will be used.
  self._init_dates(dates, freq)


...

/opt/conda/lib/python3.12/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency MS will be used.
  self._init_dates(dates, freq)
/opt/conda/lib/python3.12/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency MS will be used.
  self._init_dates(dates, freq)
/opt/conda/lib/python3.12/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency MS will be used.
  self._init_dates(dates, freq)
/opt/conda/lib/python3.12/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency MS will be used.
  self._init_dates(dates, freq)


Done: Tower Hamlets
Done: Waltham Forest


/opt/conda/lib/python3.12/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency MS will be used.
  self._init_dates(dates, freq)
/opt/conda/lib/python3.12/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency MS will be used.
  self._init_dates(dates, freq)
/opt/conda/lib/python3.12/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency MS will be used.
  self._init_dates(dates, freq)
/opt/conda/lib/python3.12/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency MS will be used.
  self._init_dates(dates, freq)


Done: Wandsworth
Done: Westminster
(32, 5)


,Borough,Forecast_2026-03,Forecast_2026-04,Forecast_2026-05,Forecast_2026-06
0,Barking and Dagenham,1699,1625,1658,1640
1,Barnet,2543,2466,2626,2533
2,Bexley,1301,1221,1238,1299
3,Brent,2608,2491,2744,2784
4,Bromley,1873,1906,1917,1871


In [14]:
# Merge actuals and forecasts
final_comparison = borough_actuals.merge(forecast_df, on="Borough")

# Reshape into long format: one row per borough per month (easier for dashboard/plotting)
rows = []
for _, row in final_comparison.iterrows():
    for month in ["2026-03", "2026-04", "2026-05", "2026-06"]:
        actual = row[f"Actual_{month}"]
        forecast = row[f"Forecast_{month}"]
        pct_error = round(abs(actual - forecast) / actual * 100, 1)
        rows.append({
            "Borough": row["Borough"],
            "Month": month,
            "Actual": actual,
            "Forecast": forecast,
            "Pct_Error": pct_error
        })

borough_actual_vs_forecast = pd.DataFrame(rows)
print(borough_actual_vs_forecast.shape)  # should be 128 rows (32 boroughs x 4 months)
print(borough_actual_vs_forecast.head(8))

borough_actual_vs_forecast.to_csv("data/borough_actual_vs_forecast_mar_jun_2026.csv", index=False)
print("Saved!")

(128, 5)
                Borough    Month  Actual  Forecast  Pct_Error
0  Barking and Dagenham  2026-03    1761      1699        3.5
1  Barking and Dagenham  2026-04    1650      1625        1.5
2  Barking and Dagenham  2026-05    1830      1658        9.4
3  Barking and Dagenham  2026-06    1870      1640       12.3
4                Barnet  2026-03    2651      2543        4.1
5                Barnet  2026-04    2352      2466        4.8
6                Barnet  2026-05    2519      2626        4.2
7                Barnet  2026-06    2590      2533        2.2
Saved!


In [16]:
borough_val = pd.read_csv("borough_forecast_validation_all32.csv")

print("ORIGINAL (summary):")
print(borough_val[borough_val["Borough"].isin(check_boroughs)])

ORIGINAL (summary):
                   Borough  Avg_Actual    MAE  Pct_Error
0   Kensington and Chelsea      1650.0   17.0        1.0
30             Westminster      5619.0  515.0        9.1
31    Richmond upon Thames       973.0  102.0       10.6
